In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


In [ ]:
df = pd.read_csv('../data/regresja/domy.csv',na_values=["?","NA",""])
df.head()

In [ ]:
print("Shape of dataframe:", df.shape)

In [ ]:
df.info()


### Usunięcie outlierów
Usunięcie outlierów zgodne z rekomendacjami autora

In [ ]:
df = df[df['GrLivArea'] <= 4000]

### Handlowanie braków danych

In [ ]:
# Categorical features where NaN means "None" (Feature does not exist)
cols_na_means_none = [
    'Alley', 'PoolQC', 'MiscFeature', 'Fence', 'FireplaceQu', 
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 
    'MasVnrType'
]

for col in cols_na_means_none:
    df[col] = df[col].fillna('None')

# Numerical features where NaN means "0" (Feature does not exist)
cols_na_means_zero = [
    'GarageArea', 'GarageCars', 
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 
    'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea'
]

for col in cols_na_means_zero:
    df[col] = df[col].fillna(0)

# Counting and drawing the gaps before filling them in
missing_columns = ['LotFrontage', 'Electrical', 'GarageYrBlt']
missing_counts = df[missing_columns].isnull().sum()

plt.figure(figsize=(8, 5))
ax = missing_counts.plot(kind='bar', color='skyblue', edgecolor='black')

ax.bar_label(ax.containers[0], padding=3)
plt.title('Number of Missing Values Before Imputation')
plt.ylabel('Count of Missing Values')
plt.xticks(rotation=0)
plt.show()

# Truly missing numerical data LotFrontage
df['LotFrontage'] = df['LotFrontage'].fillna(
    df.groupby('Neighborhood')['LotFrontage'].transform('median')
)

# Truly missing Electrical data
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

# Fill missing Garage Year Built with the Year the house was built
df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['YearBuilt'])

# Verify that all missing values have been handled
remaining_nulls = df.isnull().sum().max()
print(f"Maximum missing values in any column after cleaning: {remaining_nulls}")

In [ ]:
df.describe()

In [ ]:
df['SalePrice'].describe()

In [ ]:
df['SalePrice'].hist(bins=25)
plt.xlabel('Sale Price')
plt.ylabel('Frequency')
plt.title('Distribution of Sale Prices')
plt.show()

### Sprawdzanie czy istnieją kolumny stałe albo prawie stałe 

In [ ]:

constant_cols = []
near_constant_cols = []
id_like_cols = []

near_constant_threshold = 0.95

for col in df.columns:
    unique_count = df[col].nunique(dropna=True)

    if(unique_count <= 1):
        constant_cols.append(col)
        continue

    value_distribution = df[col].value_counts(normalize=True, dropna=False)
    top_frequency = value_distribution.iloc[0]
    top_value = value_distribution.index[0]

    if top_frequency >= near_constant_threshold:
        near_constant_cols.append((col, top_frequency, top_value, unique_count))
    
    if unique_count == len(df):
        id_like_cols.append(col)


print("Constant columns")
print(constant_cols if constant_cols else "Empty")

print("\nNear-constant columns " +f"{near_constant_threshold*100:.0f}% or more of the same value")
if near_constant_cols:
    for col, top_freq, top_value, unique_count in near_constant_cols:
        print(f"{col}: top frequency = {top_freq:.3f}, top_value = {top_value}, unique values = {unique_count}")
else:
    print("Empty")

print("\nPotential ID-like columns (every row unique)")
print(id_like_cols if id_like_cols else "Empty")

## Kodowanie zmiennych kategorycznych


In [ ]:
# Map categorical quality ratings to a numerical scale
quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
ordinal_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 
    'HeatingQC', 'KitchenQual', 'FireplaceQu', 
    'GarageQual', 'GarageCond'
]

for col in ordinal_cols:
    df[col] = df[col].map(quality_map)

# Check if there are NaNs after mapping
print(f"Not mapped:\n{df[ordinal_cols].isna().sum()}")

# Apply One-Hot Encoding to the remaining nominal categorical variables
df = pd.get_dummies(df, drop_first=True)